In [1]:
import sys
import os
sys.path.append(os.path.abspath('/acne-lds/model'))
sys.path.append(os.path.abspath('model'))
sys.path.append(os.path.abspath("/acne-lds/utils"))

In [2]:
from model_ld_smoothing import AcneModel

/opt/anaconda3/envs/pl-env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/pl-env/lib/python3.9/site-packages/torchvision/io/image.py:11: UserWarning: Failed to load image Python extension: dlopen(/opt/anaconda3/envs/pl-env/lib/python3.9/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libpng16.16.dylib
  Referenced from: <5F6B6919-410D-397C-98F2-12C5934F9DBE> /opt/anaconda3/envs/pl-env/lib/python3.9/site-packages/torchvision/image.so
  Reason: tried: '/Users/malfet/miniforge3/envs/py_39_torch-1.10.2/lib/libpng16.16.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Users/malfet/miniforge3/envs/py_39_torch-1.10.2/lib/libpng16.16.dylib' (no such file), '/Users/malfet/miniforge3/envs/py_39_torch-1.10.2/lib/libpng16.16.dylib' (no such file), '/System/Volu

In [5]:
from predict_on_img import ModelInit
from PIL import Image

model = ModelInit(path_checkpoint='model/lds-weights/model_fold_4.pth')
# img = Image.open(PATH_TO_IMAGE)
# predictions = model.predict_on_img(img)

Downloading: "https://download.pytorch.org/models/resnet50-19c8e357.pth" to /Users/utkarshsharma/.cache/torch/hub/checkpoints/resnet50-19c8e357.pth
100%|██████████| 97.8M/97.8M [00:32<00:00, 3.15MB/s]


In [48]:
# Assume ModelInit is imported and your checkpoint path is correct
import torch
model_wrapper = ModelInit(model_type="model_ld_smoothing", path_checkpoint="model/lds-weights/model_fold_4.pth", device="cpu")
model = model_wrapper.model
model.eval()

# Dummy input tensor (use actual input size expected, e.g. 3x224x224)
example_input = torch.randn(1, 3, 224, 224)
traced_model = torch.jit.trace(model, example_input)

In [128]:
import torch
from torch import nn
from collections import namedtuple

# Define a namedtuple for outputs
ModelOutput = namedtuple("ModelOutput", ["cls_pred", "lesion_count"])

class ModelWrapper(nn.Module):
    def __init__(self, model, model_type="model_ld_smoothing"):
        super().__init__()
        self.model = model
        self.model_type = model_type

    def forward(self, x):
        cls, cou, cou2cls = self.model(x)

        if self.model_type == "model_ld_smoothing":
            cls = torch.stack(
                (
                    torch.sum(cls[:, :1], 1),
                    torch.sum(cls[:, 1:4], 1),
                    torch.sum(cls[:, 4:10], 1),
                    torch.sum(cls[:, 10:], 1),
                ),
                dim=1,
            )

        cls_pred = torch.argmax(0.5 * (cls + cou2cls), dim=1)
        lesion_count = torch.argmax(cou, dim=1) + 1

        return ModelOutput(cls_pred, lesion_count)

# Wrap your model
# Initialize model (make sure the path is correct)
model_wrapper = ModelInit(model_type="model_ld_smoothing", path_checkpoint="model/lds-weights/model_fold_4.pth", device="cpu")
model = model_wrapper.model
model.eval()

wrapped_model = ModelWrapper(model)
wrapped_model.eval()
# Then trace the wrapped model
traced_model = torch.jit.trace(wrapped_model, example_input)

In [129]:
import coremltools as ct
coreml_model = ct.convert(
    traced_model,
    convert_to="neuralnetwork",
    inputs=[ct.ImageType(
        name="input_image",
        shape=(1, 3, 224, 224),
        bias=[-0.45815152, -0.361242, -0.29348266],
        scale=1/0.20132513,  # Fixed: all three channel scales
        color_layout=ct.colorlayout.RGB
    )]
)

Tuple detected at graph output. This will be flattened in the converted model.
Running MIL default pipeline:   0%|          | 0/87 [00:00<?, ? passes/s]/opt/anaconda3/envs/pl-env/lib/python3.9/site-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '910', of the source model, has been renamed to 'var_910' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
/opt/anaconda3/envs/pl-env/lib/python3.9/site-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '916', of the source model, has been renamed to 'var_916' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
Running MIL backend_neuralnetwork pipeline:   0%|          | 0/9 [00:00<?, ? passes/s]Output var var_910 of type int32 in function main is cast to type fp32
Output var var_916 of type int32 in function main is cast to type fp32
Translating MIL ==> NeuralNetwork Ops: 100%|██████████| 584/584 [00:17<00:00, 33.61 ops/

In [118]:
coreml_model_lut = ct.models.neural_network.quantization_utils.quantize_weights(
    coreml_model,
    nbits=8, 
    quantization_mode="linear_lut"
)

Quantizing using linear_lut quantization
Optimizing Neural Network before Quantization:
Finished optimizing network. Quantizing neural network..
Quantizing layer input.3 of type convolution
Quantizing layer input.11 of type convolution
Quantizing layer input.17 of type convolution
Quantizing layer out.1 of type convolution
Quantizing layer residual.1 of type convolution
Quantizing layer input.31 of type convolution
Quantizing layer input.37 of type convolution
Quantizing layer out.3 of type convolution
Quantizing layer input.49 of type convolution
Quantizing layer input.55 of type convolution
Quantizing layer out.5 of type convolution
Quantizing layer input.67 of type convolution
Quantizing layer input.73 of type convolution
Quantizing layer out.7 of type convolution
Quantizing layer residual.3 of type convolution
Quantizing layer input.87 of type convolution
Quantizing layer input.93 of type convolution
Quantizing layer out.9 of type convolution
Quantizing layer input.105 of type conv

In [119]:
coreml_model_lut.save("AcneClassQuantImpR.mlpackage")

In [120]:
img = Image.open('../data/Classification/JPEGImages/levle2_165.jpg')

In [126]:
from PIL import Image
import numpy as np
img_resized = img.resize((224, 224))

In [123]:
model =  coreml_model_lut

In [127]:
model.predict({"input_image": img_resized})

{'var_916': array([2.], dtype=float32), 'var_910': array([0.], dtype=float32)}

In [9]:
predictions = model.predict_on_img(img)

In [23]:
predictions2 = model.predict_on_img(img)

In [ ]:
import torch
preds_cls = torch.argmax(0.5 * (cls + cou2cls), dim=1)

In [30]:
import torch
preds_cls = torch.argmax(0.5 * (predictions2[0] + predictions2[2]), dim=1)